In [0]:
# Databricks notebook source
# ============================================================
# OpsIntel Copilot
# Week 6 — Security Detection Engine
#
# Purpose:
# Reads silver_security_logs and silver_admin_events
# Applies detection rules to identify suspicious activity
# Writes security alerts to Delta gold table
#
# Detection Rules:
# 1. Brute Force — same user, many failures in short window
# 2. Impossible Travel — same user, different regions < 1 hour
# 3. Privilege Escalation — sudden permission changes
# 4. Large Data Export — suspicious export activity
# 5. Config Change Near Failure — admin change before pipeline break
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

CATALOG = "workspace"
SCHEMA = "opsintel_copilot"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# ── Load silver tables ────────────────────────────────────────
security_logs = spark.table(f"{CATALOG}.{SCHEMA}.silver_security_logs")
admin_events  = spark.table(f"{CATALOG}.{SCHEMA}.silver_admin_events")

print(f"Security logs loaded: {security_logs.count()} rows")
print(f"Admin events loaded: {admin_events.count()} rows")

# COMMAND ----------
# ── Rule 1: Brute Force Detection ────────────────────────────
brute_force_alerts = (
    security_logs
    .filter(
        (F.col("is_brute_force") == True) |
        (F.col("event_type") == "brute_force")
    )
    .groupBy("user_email", "region")
    .agg(
        F.count("*").alias("event_count"),
        F.min("event_time").alias("first_seen"),
        F.max("event_time").alias("last_seen")
    )
    .filter(F.col("event_count") >= 1)
    .withColumn("alert_type", F.lit("brute_force"))
    .withColumn("severity", F.lit("high"))
    .withColumn("confidence_score", F.lit(0.90))
    .withColumn("description", F.concat(
        F.lit("Brute force detected: "),
        F.col("event_count").cast("string"),
        F.lit(" attempts from user "),
        F.col("user_email")
    ))
    .withColumn("alert_id", F.concat(
        F.lit("BF_"), F.col("user_email"), F.lit("_"), F.col("region")
    ))
    .withColumn("detected_at", F.current_timestamp())
)

print(f"Brute force alerts: {brute_force_alerts.count()}")

# COMMAND ----------
# ── Rule 2: Impossible Travel Detection ──────────────────────
impossible_travel_alerts = (
    security_logs
    .filter(
        (F.col("is_impossible_travel") == True) |
        (F.col("event_type") == "impossible_travel")
    )
    .groupBy("user_email")
    .agg(
        F.count("*").alias("event_count"),
        F.collect_set("region").alias("regions_visited"),
        F.min("event_time").alias("first_seen"),
        F.max("event_time").alias("last_seen")
    )
    .filter(F.col("event_count") >= 1)
    .withColumn("alert_type", F.lit("impossible_travel"))
    .withColumn("severity", F.lit("critical"))
    .withColumn("confidence_score", F.lit(0.95))
    .withColumn("region", F.lit("multiple"))
    .withColumn("description", F.concat(
        F.lit("Impossible travel detected for user "),
        F.col("user_email"),
        F.lit(" across multiple regions")
    ))
    .withColumn("alert_id", F.concat(F.lit("IT_"), F.col("user_email")))
    .withColumn("detected_at", F.current_timestamp())
    .drop("regions_visited")
)

print(f"Impossible travel alerts: {impossible_travel_alerts.count()}")

# COMMAND ----------
# ── Rule 3: Privilege Escalation Detection ───────────────────
escalation_alerts = (
    security_logs
    .filter(
        (F.col("is_escalation") == True) |
        (F.col("event_type") == "privilege_escalation")
    )
    .groupBy("user_email", "region")
    .agg(
        F.count("*").alias("event_count"),
        F.min("event_time").alias("first_seen"),
        F.max("event_time").alias("last_seen")
    )
    .withColumn("alert_type", F.lit("privilege_escalation"))
    .withColumn("severity", F.lit("critical"))
    .withColumn("confidence_score", F.lit(0.92))
    .withColumn("description", F.concat(
        F.lit("Privilege escalation detected for user "),
        F.col("user_email")
    ))
    .withColumn("alert_id", F.concat(
        F.lit("PE_"), F.col("user_email"), F.lit("_"), F.col("region")
    ))
    .withColumn("detected_at", F.current_timestamp())
)

print(f"Privilege escalation alerts: {escalation_alerts.count()}")

# COMMAND ----------
# ── Rule 4: Large Data Export Detection ──────────────────────
export_alerts = (
    security_logs
    .filter(
        (F.col("is_large_export") == True) |
        (F.col("event_type") == "large_data_export")
    )
    .groupBy("user_email", "region")
    .agg(
        F.count("*").alias("event_count"),
        F.min("event_time").alias("first_seen"),
        F.max("event_time").alias("last_seen")
    )
    .withColumn("alert_type", F.lit("large_data_export"))
    .withColumn("severity", F.lit("high"))
    .withColumn("confidence_score", F.lit(0.85))
    .withColumn("description", F.concat(
        F.lit("Suspicious large data export by user "),
        F.col("user_email")
    ))
    .withColumn("alert_id", F.concat(
        F.lit("LDE_"), F.col("user_email"), F.lit("_"), F.col("region")
    ))
    .withColumn("detected_at", F.current_timestamp())
)

print(f"Large data export alerts: {export_alerts.count()}")

# COMMAND ----------
# ── Rule 5: Config Change Detection ──────────────────────────
config_change_alerts = (
    admin_events
    .filter(
        (F.col("action") == "config_changed") |
        (F.col("is_config_change") == True)
    )
    .groupBy("admin_user", "target_service", "region")
    .agg(
        F.count("*").alias("event_count"),
        F.min("event_time").alias("first_seen"),
        F.max("event_time").alias("last_seen")
    )
    .withColumn("alert_type", F.lit("suspicious_config_change"))
    .withColumn("severity", F.lit("high"))
    .withColumn("confidence_score", F.lit(0.80))
    .withColumn("description", F.concat(
        F.lit("Critical config change on "),
        F.col("target_service"),
        F.lit(" by "),
        F.col("admin_user")
    ))
    .withColumn("alert_id", F.concat(
        F.lit("CC_"), F.col("admin_user"), F.lit("_"), F.col("target_service")
    ))
    .withColumn("detected_at", F.current_timestamp())
    .withColumnRenamed("admin_user", "user_email")
)

print(f"Config change alerts: {config_change_alerts.count()}")

# COMMAND ----------
# ── Combine all alerts ────────────────────────────────────────
common_cols = [
    "alert_id", "alert_type", "user_email", "region",
    "severity", "confidence_score", "description",
    "event_count", "first_seen", "last_seen", "detected_at"
]

all_alerts = (
    brute_force_alerts.select(common_cols)
    .union(impossible_travel_alerts.select(common_cols))
    .union(escalation_alerts.select(common_cols))
    .union(export_alerts.select(common_cols))
    .union(config_change_alerts.select(common_cols))
)

print(f"\nTotal security alerts detected: {all_alerts.count()}")
display(all_alerts)

# COMMAND ----------
# ── Write to Delta table ──────────────────────────────────────
ALERTS_PATH = "s3://opsintel-copilot-angad-0025/gold/security_alerts/"

(
    all_alerts.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(ALERTS_PATH)
)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.security_alerts_detected")
spark.sql(f"""
    CREATE TABLE {CATALOG}.{SCHEMA}.security_alerts_detected
    USING DELTA
    LOCATION '{ALERTS_PATH}'
""")

print(f"Security alerts written to: {CATALOG}.{SCHEMA}.security_alerts_detected")
print(f"Total alerts: {spark.table(f'{CATALOG}.{SCHEMA}.security_alerts_detected').count()}")